# H&M 거래·상품 이미지 표본 추출

비공개 Kaggle Notebook에서 **Add Input**으로 H&M Personalized Fashion Recommendations 대회를 연결한 뒤 실행한다. 대회 입력은 Kaggle에서 읽고, 로컬로 옮길 파일은 이용 조건을 확인한 후 `hm_sample.zip` 하나만 선택한다. 표본 ZIP과 해제한 파일은 로컬 프로젝트의 `data/raw/hm/`에 둔다.

기본 기간은 2020년 6~8월이며 원본이 이 기간을 포함하는지 실행 중 검사한다. 초깃값은 고객 최대 1,000명과 실제 상품 이미지 최대 200장이다. 채널 코드가 확인될 때까지 두 채널을 함께 사용한다. 구매일과 고객 ID, 원래 가격을 보존하며, RFM 기준일은 표본 추출 전 범위에서 결정한다.

In [ ]:
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
DATASET_DIR = None  # Set to a Path if Kaggle shows more than one matching input.
OUTPUT_DIR = Path("/kaggle/working")

START_DATE = "2020-06-01"
END_DATE = "2020-09-01"  # Exclusive.
SALES_CHANNEL = None  # Use both channels until their meanings are verified.
SEED = 42
MAX_CUSTOMERS = 1_000
MAX_IMAGES = 200
MIN_TRANSACTIONS = 1_000
CHUNK_SIZE = 250_000

In [ ]:
import json
from collections import Counter
from tempfile import TemporaryDirectory
from zipfile import ZIP_DEFLATED, ZIP_STORED, ZipFile

import numpy as np
import pandas as pd


TRANSACTION_COLUMNS = ["t_dat", "customer_id", "article_id", "price", "sales_channel_id"]
ARTICLE_COLUMNS = ["article_id", "prod_name", "product_type_name"]


def find_dataset(input_root):
    candidates = [
        path.parent
        for pattern in ("transactions_train.csv", "*/transactions_train.csv", "*/*/transactions_train.csv")
        for path in input_root.glob(pattern)
        if (path.parent / "articles.csv").is_file() and (path.parent / "images").is_dir()
    ]
    if len(candidates) != 1:
        raise ValueError("Attach the H&M competition input or set DATASET_DIR to its folder.")
    return candidates[0]


def transaction_chunks(path, start, end, sales_channel, chunk_size):
    for rows in pd.read_csv(path, usecols=TRANSACTION_COLUMNS, dtype="string", chunksize=chunk_size):
        rows["customer_id"] = rows["customer_id"].str.strip().replace("", pd.NA)
        rows["article_id"] = rows["article_id"].str.strip().replace("", pd.NA)
        dates = pd.to_datetime(rows["t_dat"], format="%Y-%m-%d", errors="coerce")
        amounts = pd.to_numeric(rows["price"], errors="coerce").to_numpy(dtype=float, na_value=np.nan)
        customer_ok = rows["customer_id"].notna()
        date_ok = dates.notna()
        amount_ok = np.isfinite(amounts) & (amounts > 0)
        valid = customer_ok & date_ok & amount_ok
        in_scope = dates.ge(start) & dates.lt(end)
        if sales_channel is not None:
            in_scope &= rows["sales_channel_id"].eq(str(sales_channel)).fillna(False)
        audit = {
            "source_rows": len(rows),
            "missing_customer_rows": int((~customer_ok).sum()),
            "invalid_date_rows": int((customer_ok & ~date_ok).sum()),
            "invalid_amount_rows": int((customer_ok & date_ok & ~amount_ok).sum()),
            "out_of_scope_rows": int((valid & ~in_scope).sum()),
        }
        selected = valid & in_scope
        yield rows.loc[selected, TRANSACTION_COLUMNS], dates.loc[selected], dates.min(), dates.max(), audit


def choose(values, limit, seed):
    ordered = sorted(values)
    indices = np.random.default_rng(seed).choice(len(ordered), size=min(limit, len(ordered)), replace=False)
    return {ordered[index] for index in indices}


def prepare_hm_sample(dataset_dir, output_dir, *, start, end, seed, max_customers,
                      max_images, min_transactions, chunk_size, sales_channel=None):
    start, end = pd.Timestamp(start), pd.Timestamp(end)
    if not (start.is_month_start and end.is_month_start and start + pd.DateOffset(months=3) <= end):
        raise ValueError("Use month starts for an interval of at least three complete months.")
    if min(max_customers, max_images, min_transactions, chunk_size) < 1:
        raise ValueError("Sampling limits and chunk size must be positive.")

    transaction_path = dataset_dir / "transactions_train.csv"
    customers, covered_months = set(), set()
    counts = Counter()
    reference_date = source_start = source_end = None
    for rows, dates, raw_start, raw_end, audit in transaction_chunks(
        transaction_path, start, end, sales_channel, chunk_size
    ):
        counts.update(audit)
        if pd.notna(raw_start):
            source_start = raw_start if source_start is None else min(source_start, raw_start)
            source_end = raw_end if source_end is None else max(source_end, raw_end)
        if not rows.empty:
            customers.update(rows["customer_id"].unique())
            covered_months.update(dates.dt.to_period("M").unique())
            reference_date = dates.max() if reference_date is None else max(reference_date, dates.max())
            counts["eligible_rows"] += len(rows)

    expected_months = set(pd.period_range(start, end - pd.Timedelta(days=1), freq="M"))
    if (source_start is None or source_start > start or source_end < end - pd.Timedelta(days=1)
            or covered_months != expected_months):
        raise ValueError("The source and channel do not cover the configured complete months.")
    chosen_customers = choose(customers, max_customers, seed)
    population_size = len(customers)
    del customers

    output_dir.mkdir(parents=True, exist_ok=True)
    destination = output_dir / "hm_sample.zip"
    with TemporaryDirectory(dir=output_dir) as temp_dir:
        work = Path(temp_dir)
        sample_csv = work / "transactions_sample.csv"
        transaction_count = 0
        article_ids = set()
        for rows, _, _, _, _ in transaction_chunks(transaction_path, start, end, sales_channel, chunk_size):
            sample = rows.loc[rows["customer_id"].isin(chosen_customers)]
            if sample.empty:
                continue
            sample.to_csv(sample_csv, mode="a", header=transaction_count == 0, index=False)
            transaction_count += len(sample)
            article_ids.update(sample["article_id"].dropna().unique())
        if transaction_count < min_transactions:
            raise ValueError(f"Only {transaction_count} eligible sample rows. Increase MAX_CUSTOMERS or the period.")

        articles = pd.read_csv(dataset_dir / "articles.csv", usecols=ARTICLE_COLUMNS, dtype="string")
        articles = articles.loc[articles["article_id"].isin(article_ids)].copy()
        if articles["article_id"].duplicated().any():
            raise ValueError("Duplicate product IDs in the selected article metadata.")
        image_paths = {
            article_id: dataset_dir / "images" / article_id[:3] / f"{article_id}.jpg"
            for article_id in articles["article_id"]
            if article_id.isascii() and article_id.isdecimal()
        }
        image_paths = {article_id: path for article_id, path in image_paths.items() if path.is_file()}
        if not image_paths:
            raise ValueError("No product images matched the selected transactions and metadata.")
        chosen_images = choose(image_paths, max_images, seed)
        articles["source_image_available"] = articles["article_id"].isin(image_paths)
        articles["image_sampled"] = articles["article_id"].isin(chosen_images)
        articles.to_csv(work / "articles_sample.csv", index=False)

        counts["sampled_rows"] = transaction_count
        counts["sample_excluded_rows"] = counts["eligible_rows"] - transaction_count
        excluded = sum(counts[name] for name in (
            "missing_customer_rows", "invalid_date_rows", "invalid_amount_rows",
            "out_of_scope_rows", "sample_excluded_rows",
        ))
        if counts["sample_excluded_rows"] < 0 or counts["source_rows"] != excluded + transaction_count:
            raise ValueError("Transaction counts could not be reconciled.")
        manifest = {
            "source": "h-and-m-personalized-fashion-recommendations",
            "source_start": str(source_start.date()),
            "source_end": str(source_end.date()),
            "start_inclusive": str(start.date()),
            "end_exclusive": str(end.date()),
            "sales_channel": sales_channel,
            "seed": seed,
            "reference_date": str(reference_date.date()),
            "eligible_customers": population_size,
            "sampled_customers": len(chosen_customers),
            "sampled_product_ids": len(article_ids),
            "matched_products": len(articles),
            "matched_products_without_images": len(articles) - len(image_paths),
            "products_with_images_outside_image_sample": len(image_paths) - len(chosen_images),
            "sampled_images": len(chosen_images),
            "transactions": dict(counts),
        }
        (work / "sample_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
        archive_path = work / destination.name
        with ZipFile(archive_path, "w", compression=ZIP_DEFLATED) as archive:
            for name in ("transactions_sample.csv", "articles_sample.csv", "sample_manifest.json"):
                archive.write(work / name, name)
            for article_id in sorted(chosen_images):
                path = image_paths[article_id]
                archive.write(path, f"images/{path.parent.name}/{path.name}", compress_type=ZIP_STORED)
        archive_path.replace(destination)
    return destination, manifest

In [ ]:
dataset_dir = DATASET_DIR if DATASET_DIR is not None else find_dataset(INPUT_ROOT)
archive, manifest = prepare_hm_sample(
    dataset_dir, OUTPUT_DIR,
    start=START_DATE, end=END_DATE, seed=SEED,
    max_customers=MAX_CUSTOMERS, max_images=MAX_IMAGES,
    min_transactions=MIN_TRANSACTIONS, chunk_size=CHUNK_SIZE,
    sales_channel=SALES_CHANNEL,
)
print(f"Archive: {archive.name} ({archive.stat().st_size / (1024 ** 2):.2f} MiB)")
print(f"Rows: {manifest['transactions']['sampled_rows']:,}")
print(f"Customers: {manifest['sampled_customers']:,}")
print(f"Product images: {manifest['sampled_images']:,}")
print(f"RFM reference date: {manifest['reference_date']}")